# Normalization And Feature Selection

**REQUIRED DAY 2**

## Why normalize at all

Two cells can have genuinely different total RNA content just from cell size or capture efficiency, not biology. Without correcting for that, "this cell has more transcripts of gene X" might just mean "this cell was captured with more RNA overall." Normalization puts cells on a comparable scale before you compare them.

## Load your checkpoint

This is a fresh kernel -- loading back the `adata` you saved at the end of [05_loading_data_and_qc.ipynb](05_loading_data_and_qc.ipynb), QC-filtered version included if you did your own filtering there.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/checkpoint_05_qc.h5ad")
adata


Before normalizing, keep the raw counts around under an explicit name — you'll want them later, and once `adata.X` is overwritten there's no getting the original values back except by reloading. Save a copy into `adata.layers["counts"]` (any name works, but "counts" is the scanpy convention).

Then normalize: look up scanpy's per-cell total-count normalization function ([`sc.pp.normalize_total`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.normalize_total.html) — use `target_sum=1e4`, the standard "counts per 10k" convention) and its log-transform function ([`sc.pp.log1p`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.log1p.html)). Both modify `adata.X` in place.

In [ ]:
## Fill in:
## 1) adata.layers["counts"] = adata.X.copy()
## 2) sc.pp.normalize_total(adata, target_sum=1e4)
## 3) sc.pp.log1p(adata)




`normalize_total` scales each cell's counts so total counts per cell are comparable; `log1p` (log(1+x)) compresses the wide dynamic range of expression values so a handful of very highly expressed genes don't dominate every downstream step.

## The bug that's easy to introduce here

Once you have both `adata.layers["counts"]` (raw) and `adata.X` (normalized, log-transformed), **it's a real, common mistake to accidentally feed the wrong one into a downstream step** — e.g., running a statistical test that assumes raw counts on the log-transformed layer, or computing a "fold change" on already-log-transformed values without accounting for that. Whoever writes that calculation next — you, or anyone else — needs to be told explicitly which layer to use; it's not something to leave implicit.

## Feature selection: not every gene is useful for clustering

Most genes are either not expressed or don't vary meaningfully across cells in a given dataset — including them in downstream steps like PCA mostly adds noise. Highly variable gene (HVG) selection keeps the genes that actually carry information about differences between cells. Look up scanpy's function ([`sc.pp.highly_variable_genes`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.highly_variable_genes.html) — use `n_top_genes=2000`) and its matching plot function ([`sc.pl.highly_variable_genes`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pl.highly_variable_genes.html)).

In [ ]:
## Fill in: sc.pp.highly_variable_genes(adata, n_top_genes=2000), then sc.pl.highly_variable_genes(adata)



This doesn't delete the other genes — `adata.var["highly_variable"]` flags them, and steps like PCA use the flag by default while `rank_genes_groups` in [08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb) can still test every gene, not just the HVG subset.

## Save your checkpoint

[07_dimensionality_reduction_and_clustering.ipynb](07_dimensionality_reduction_and_clustering.ipynb) loads this back in.

In [ ]:
adata.write_h5ad("results/checkpoint_06_normalized.h5ad")
print("Saved to results/checkpoint_06_normalized.h5ad")


## Practice

Compute the mean expression of one gene (pick any gene you've seen mentioned, e.g. `LYZ`) two ways -- once from `adata.layers["counts"]` (raw) and once from `adata.X` (normalized, log-transformed), in the cell below. They will not be the same number. Write one sentence on which one you'd actually use for a fold-change calculation between two groups of cells, and why.

In [ ]:
# Compute mean expression of your chosen gene from both layers here, and note which one you'd use and why.




## Further reading

- [Single-cell best practices — Normalization](https://www.sc-best-practices.org/preprocessing_visualization/normalization.html)
- [Single-cell best practices — Feature Selection](https://www.sc-best-practices.org/preprocessing_visualization/feature_selection.html)
- [Seurat's `NormalizeData`/`FindVariableFeatures`](https://satijalab.org/seurat/articles/essential_commands.html) — the R/Seurat equivalents of both steps in this notebook, if you're curious.